**Завдання 5. Ноутбук 04 — снепшот поточних курсів**

In [1]:
!pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 203.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 190.6 MB/s  0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.80.0
    Uninstalling grpcio-1.80.0:
      Successfully uninstalled grpcio-1.80.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [google-cloud-bigquery]


In [2]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "project-nbu"   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-nbu


**Завдання 5.1.** Прочитайте nbu_raw.raw_rates, розгорніть payload у колонки і нормалізуйте значення так само, як у завданні 3.

In [3]:
# --- ЗАВДАННЯ 5.1: Чтение, развертывание и нормализация данных для снепшота ---

# 1. Читаем необходимые колонки из слоя Bronze
query_bronze = f"""
SELECT ingested_at, business_date, payload
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""
df_bronze = client.query(query_bronze).to_dataframe()

# 2. Разворачиваем JSON-текст из payload в отдельные колонки
import json
parsed_snapshot = pd.json_normalize(df_bronze["payload"].map(json.loads))

# Добавляем системное время ingested_at и business_date из бронзового DataFrame
parsed_snapshot["ingested_at"] = df_bronze["ingested_at"]
parsed_snapshot["business_date"] = df_bronze["business_date"]

# 3. Нормализуем значения (как в задании 3)
parsed_snapshot["cc"] = parsed_snapshot["cc"].str.strip().str.upper()
parsed_snapshot["txt"] = parsed_snapshot["txt"].str.strip()
parsed_snapshot["r030"] = parsed_snapshot["r030"].astype("Int64")
parsed_snapshot["rate"] = parsed_snapshot["rate"].astype(float)

# Проверка: выводим размер получившегося набора данных
print(f"Данные успешно прочитаны и нормализованы. Строк: {len(parsed_snapshot)}")
print(parsed_snapshot[["cc", "txt", "rate", "business_date"]].head(3))

Данные успешно прочитаны и нормализованы. Строк: 90
    cc                   txt      rate business_date
0  DZD      Алжирський динар   0.33615    2026-08-25
1  AUD  Австралійський долар  32.03210    2026-08-25
2  BDT                  Така   0.36460    2026-08-25


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 5.2.** Для кожної валюти залиште рядок із максимальною business_date. Якщо за одну дату є кілька записів (bronze дописується), беріть найсвіжіший за ingested_at.

In [4]:
# --- ЗАВДАННЯ 5.2: Фильтрация самых актуальных курсов для снепшота ---

# Выполняем сортировку по дате курса и времени записи, оставляя последнюю запись
snap = (
    parsed_snapshot.sort_values(["business_date", "ingested_at"])
    .drop_duplicates(subset="cc", keep="last")
    .copy()
)

# Проверка: количество строк в снепшоте должно быть равно количеству уникальных валют (~60)
print(f"Количество валют в актуальном снепшоте: {len(snap)}")
print("\nФрагмент полученного снепшота:")
print(snap[["cc", "rate", "business_date", "ingested_at"]].head(3))

Количество валют в актуальном снепшоте: 45

Фрагмент полученного снепшота:
     cc      rate business_date                      ingested_at
45  DZD   0.33615    2026-08-25 2026-08-24 13:46:45.990685+00:00
46  AUD  32.03210    2026-08-25 2026-08-24 13:46:45.990685+00:00
47  BDT   0.36460    2026-08-25 2026-08-24 13:46:45.990685+00:00


**Завдання 5.3.** Приєднайте currency_key із таблиці nbu_dwh.dim_currency — не вигадуйте ключі заново. Валютам, яких немає у вимірі, поставте -1.

In [5]:
# --- ЗАВДАННЯ 5.3: Приєднання сурогатних ключів валют із dim_currency ---

# 1. Читаємо актуальний вимір валют із шару Gold
query_dim_curr = f"SELECT currency_key, currency_code FROM `{PROJECT_ID}.nbu_dwh.dim_currency`"
df_dim_currency = client.query(query_dim_curr).to_dataframe()

# 2. Приєднуємо вимір до нашого снепшоту через Left Join
snap_with_keys = pd.merge(
    snap, 
    df_dim_currency, 
    left_on="cc", 
    right_on="currency_code", 
    how="left"
)

# 3. Якщо якась валюта не знайшлася у вимірі, ставимо технічний ключ -1
snap_with_keys["currency_key"] = snap_with_keys["currency_key"].fillna(-1).astype(int)

# Перевірка: виводимо перші 3 рядки для контролю наявності ключів
print("--- Фрагмент слітого снепшоту з ключами валют ---")
print(snap_with_keys[["currency_key", "cc", "rate", "business_date"]].head(3))

# Перевірка наявності помилкових або невідомих валют (ключ -1)
unknown_count = (snap_with_keys["currency_key"] == -1).sum()
print(f"\nКількість валют із невідомим ключем (-1): {unknown_count}")

--- Фрагмент слітого снепшоту з ключами валют ---
   currency_key   cc      rate business_date
0            10  DZD   0.33615    2026-08-25
1             2  AUD  32.03210    2026-08-25
2             4  BDT   0.36460    2026-08-25

Кількість валют із невідомим ключем (-1): 0


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 5.4.** Додайте колонку snapshot_ts — момент побудови снепшоту (pd.Timestamp.now(tz="UTC")).
Отримати на виході: currency_key, currency_code, currency_name, rate, business_date, snapshot_ts

In [6]:
# --- ЗАВДАННЯ 5.4: Додавання мітки часу та відбір фінальних колонок ---

# 1. Додаємо колонку snapshot_ts із точним часом побудови снепшоту в UTC
snap_with_keys["snapshot_ts"] = pd.Timestamp.now(tz="UTC")

# 2. Перейменовуємо оригінальне поле 'txt' у 'currency_name' для відповідності вимогам
snap_final = snap_with_keys.rename(columns={"txt": "currency_name"})

# 3. Відбираємо стовпчики у строго визначеному порядку
target_cols = [
    "currency_key", "currency_code", "currency_name", 
    "rate", "business_date", "snapshot_ts"
]
snap_final = snap_final[target_cols]

# Перевірка: виводимо типи полів та перші 3 рядки готового снепшоту
print("--- Схема фінальних колонок снепшоту ---")
print(snap_final.dtypes)

print("\n--- Фрагмент готової таблиці для запису ---")
print(snap_final.head(3))

--- Схема фінальних колонок снепшоту ---
currency_key                   int64
currency_code                 object
currency_name                 object
rate                         float64
business_date                 dbdate
snapshot_ts      datetime64[us, UTC]
dtype: object

--- Фрагмент готової таблиці для запису ---
   currency_key currency_code         currency_name      rate business_date  \
0            10           DZD      Алжирський динар   0.33615    2026-08-25   
1             2           AUD  Австралійський долар  32.03210    2026-08-25   
2             4           BDT                  Така   0.36460    2026-08-25   

                       snapshot_ts  
0 2026-08-27 14:27:26.558314+00:00  
1 2026-08-27 14:27:26.558314+00:00  
2 2026-08-27 14:27:26.558314+00:00  


**Завдання 5.5.** Запишіть у nbu_dwh.snapshot_rates_current у режимі WRITE_TRUNCATE.

In [7]:
# --- ЗАВДАННЯ 5.5: Запис поточного снепшоту в шар Gold ---

TABLE_ID = f"{PROJECT_ID}.nbu_dwh.snapshot_rates_current"

# Налаштовуємо конфігурацію: повний перезапис таблиці (WRITE_TRUNCATE)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Завантажуємо DataFrame у BigQuery
job = client.load_table_from_dataframe(snap_final, TABLE_ID, job_config=job_config)
job.result()  # Очікуємо завершення операції

print(f"✅ Поточний снепшот успішно збережено в таблицю {TABLE_ID}!")
print(f"Кількість записаних валют: {len(snap_final)}")

✅ Поточний снепшот успішно збережено в таблицю project-nbu.nbu_dwh.snapshot_rates_current!
Кількість записаних валют: 45


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


**Завдання 5.6.** Перевірте й виведіть результат: рядків стільки, скільки унікальних валют у bronze; currency_code унікальний; немає жодного currency_key = -1.

In [8]:
# --- ЗАВДАННЯ 5.6: Фінальна валідація поточного снепшоту курсів ---

# 1. Перевірка: чи дорівнює кількість рядків кількості унікальних валют у bronze
expected_cc_cnt = parsed_snapshot["cc"].nunique()
count_matches = len(snap_final) == expected_cc_cnt

# 2. Перевірка: чи є унікальним стовпчик currency_code у фінальній таблиці
is_cc_unique = snap_final["currency_code"].is_unique

# 3. Перевірка: чи дійсно немає жодного невідомого ключа currency_key = -1
no_unknown_keys = not (snap_final["currency_key"] == -1).any()

# Виведення результатів трьох перевірок
print(count_matches)
print(is_cc_unique)
print(no_unknown_keys)

True
True
True
